# 🤖 Model Training — Manufacturing Predictive Maintenance

Trains a **SynapseML LightGBM** classifier to predict equipment maintenance needs
based on sensor readings and production metrics.

**Target:** `needs_maintenance` (1 = downtime > 60 min, likely needs maintenance)

**Reads from:** `gold_ml_features`

**Writes to:** `gold_ml_model_metrics`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load feature table
features_df = spark.read.table('gold_ml_features')
print(f'Feature rows: {features_df.count():,}')
print(f'Target distribution:')
features_df.groupBy('needs_maintenance').count().show()

In [ ]:
# Define feature columns (numeric only for LightGBM)
numeric_features = [
    'avg_temp', 'std_temp', 'max_temp', 'min_temp', 'temp_range',
    'avg_pressure', 'std_pressure', 'max_pressure',
    'avg_vibration', 'std_vibration', 'max_vibration',
    'avg_humidity',
    'reading_count', 'temp_anomaly_count', 'vib_anomaly_count', 'anomaly_ratio',
    'total_units', 'total_defects', 'total_downtime',
    'batch_count', 'avg_yield', 'avg_defect_rate',
    'equipment_age_days',
]

# Index categorical columns
indexer_type = StringIndexer(inputCol='machine_type', outputCol='machine_type_idx', handleInvalid='keep')
indexer_line = StringIndexer(inputCol='production_line', outputCol='production_line_idx', handleInvalid='keep')

indexed_df = indexer_type.fit(features_df).transform(features_df)
indexed_df = indexer_line.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + ['machine_type_idx', 'production_line_idx']

# Assemble feature vector
assembler = VectorAssembler(
    inputCols=all_features,
    outputCol='features',
    handleInvalid='skip'
)
model_df = assembler.transform(indexed_df).select('features', col('needs_maintenance').alias('label'))
print(f'Model-ready rows: {model_df.count():,}')
print(f'Feature count: {len(all_features)}')

In [ ]:
# Train/test split (80/20)
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} rows')
print(f'Test:  {test_df.count():,} rows')

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier

# Train LightGBM classifier
lgbm = LightGBMClassifier(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    rawPredictionCol='rawPrediction',
    probabilityCol='probability',
    numLeaves=31,
    numIterations=200,
    learningRate=0.05,
    featureFraction=0.8,
    baggingFraction=0.8,
    baggingFreq=5,
    objective='binary',
    isUnbalance=True,  # handles class imbalance
)

model = lgbm.fit(train_df)
print('LightGBM model trained')

In [ ]:
# Evaluate on test set
predictions = model.transform(test_df)

# AUC-ROC
auc_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
auc = auc_eval.evaluate(predictions)

# AUC-PR
pr_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderPR')
auc_pr = pr_eval.evaluate(predictions)

# Accuracy & F1
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
accuracy = acc_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)

print(f'\n=== Model Evaluation ===')
print(f'AUC-ROC:  {auc:.4f}')
print(f'AUC-PR:   {auc_pr:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')

In [ ]:
# Save model metrics to Gold table
metrics_data = [
    ('manufacturing-qc', 'predictive-maintenance', 'LightGBMClassifier',
     len(all_features), train_df.count(), test_df.count(),
     float(auc), float(auc_pr), float(accuracy), float(f1))
]
metrics_df = spark.createDataFrame(
    metrics_data,
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'auc_pr', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved to gold_ml_model_metrics')
metrics_df.show(truncate=False)

In [ ]:
# Save the trained model for scoring notebook
model.save('Files/models/predictive_maintenance_lgbm')
print('Model saved to Files/models/predictive_maintenance_lgbm')